# ДЗ 2 — Fine-tuning DETR на COCO-subset (Google Colab)

Обёртка для запуска пайплайна из `src/` на бесплатном GPU Colab (T4).

**Что делает:** загружает маленький detection-датасет (CPPE-5, COCO-формат), дообучает `facebook/detr-resnet-50`, логирует loss в TensorBoard, снимает trace профайлера, сохраняет чекпойнты, строит графики потерь / mAP и проводит error analysis.

**Перед запуском:** `Runtime → Change runtime type → Hardware accelerator → GPU`.

Порядок: выполняйте ячейки сверху вниз (`Shift+Enter`).

## 1. Проверка GPU

In [ ]:
!nvidia-smi -L || echo 'GPU не найден — Runtime > Change runtime type > GPU'
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 2. Получить код

Впишите URL своего GitHub-репозитория в `REPO_URL`. Если репозитория ещё нет — загрузите папку `src/` вручную через панель **Files** в `/content/hw3` и оставьте `REPO_URL` пустым.

In [ ]:
import os

REPO_URL = ""  # напр. https://github.com/<user>/<repo>.git
WORK = "/content/hw3"

if REPO_URL:
    !git clone $REPO_URL $WORK
else:
    os.makedirs(WORK, exist_ok=True)
    print("REPO_URL пуст: загрузите src/ в", WORK, "через панель Files,")
    print("либо впишите REPO_URL и перезапустите ячейку.")

os.chdir(WORK)
print("CWD:", os.getcwd())
!ls -la

## 3. Установка зависимостей

В Colab уже есть `torch`, `torchvision`, `matplotlib`, `pillow`, `scipy` — доустанавливаем остальное.

In [ ]:
!pip -q install "transformers>=4.40" "datasets>=2.18" timm torchmetrics pycocotools tensorboard
print('done')

## 4. Обучение

На GPU можно позволить полный resize и больше эпох. Выберите один из вариантов.

In [ ]:
# --- Быстрое демо (несколько минут на T4) ---
!python src/train.py --train-size 300 --val-size 100 --epochs 12 --batch-size 4 --image-short 600 --image-long 1000

# --- Полный прогон (дольше, выше качество) — раскомментируйте ---
# !python src/train.py --epochs 30 --batch-size 4 --image-short 800 --image-long 1333

## 5. Таблица метрик (mAP / mAP50)

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('outputs/metrics.md').read()))

## 6. Графики потерь и mAP

In [ ]:
Image('outputs/figures/loss_and_map.png')

## 7. Error analysis

`--threshold` подбирайте под уверенность модели: после короткого обучения score низкие (0.1–0.3), после полного прогона можно 0.5–0.7.

In [ ]:
!python src/error_analysis.py --ckpt outputs/checkpoints/best.pt --val-size 100 --threshold 0.3 --n-viz 8

In [ ]:
display(Markdown(open('outputs/error_analysis.md').read()))
display(Image('outputs/figures/error_analysis.png'))

### Предсказанные боксы (красные) vs ground truth (зелёные пунктирные)

In [ ]:
Image('outputs/figures/predictions.png')

## 8. TensorBoard (loss-кривые по шагам, mAP по эпохам)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir outputs/runs

## 9. (Опционально) Скачать артефакты

Упаковать чекпойнт, графики, метрики, trace профайлера и логи в один архив.

In [ ]:
import shutil
shutil.make_archive('/content/detr_outputs', 'zip', 'outputs')
from google.colab import files
files.download('/content/detr_outputs.zip')